# Day 6 — Hugging Face Hub Deployment
**Goal:** Push the fine-tuned NyayaGPT adapter + a detailed model card to HF Hub.

**Two modes:**
1. **Adapter-only** (~1 GB for 7B) — fast, users need base model at runtime
2. **Merged FP16 model** (~14 GB) — standalone, larger upload

In [1]:
import sys, os, subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.chdir(PROJECT_ROOT)

In [7]:
# ── Step 1: Authenticate ──────────────────────────────────────────────────────
# Run in terminal first: huggingface-cli login
# Or set token here:

HF_TOKEN = os.getenv('HF_TOKEN', '')  # set via: export HF_TOKEN=hf_XXXX
HF_USERNAME = 'gauravgarwal'      # <- CHANGE THIS

if not HF_TOKEN:
    print('⚠ HF_TOKEN not set — will use huggingface-cli credentials')
    print('  Run: huggingface-cli login')
else:
    print(f'✓ HF_TOKEN set ({HF_TOKEN[:8]}…)')

print(f'\nWill deploy to: {HF_USERNAME}/NyayaGPT-Mistral7B-adapter')

⚠ HF_TOKEN not set — will use huggingface-cli credentials
  Run: huggingface-cli login

Will deploy to: gauravgarwal/NyayaGPT-Mistral7B-adapter


In [ ]:
from huggingface_hub import login
login(token="")  # or leave empty for interactive prompt

In [4]:
# ── Step 2: Load eval results for model card ──────────────────────────────────
import json
from nyaya_pipeline import config

eval_results = {}
benchmark_results = []

# Try to load evaluation results from MLflow
try:
    import mlflow
    mlflow.set_tracking_uri(config.MLFLOW_URI)
    runs = mlflow.search_runs(experiment_names=[config.MLFLOW_EXPERIMENT_EVAL])
    if not runs.empty:
        latest = runs.iloc[0]
        eval_results = {
            'finetuned_rouge1':  latest.get('metrics.finetuned_rouge1', 'N/A'),
            'finetuned_rougeL':  latest.get('metrics.finetuned_rougeL', 'N/A'),
            'finetuned_ragas_faithfulness': latest.get('metrics.finetuned_ragas_faithfulness', 'N/A'),
        }
        print('✓ Eval results loaded from MLflow')
except Exception as e:
    print(f'⚠ Could not load MLflow results: {e}')

# Try to load benchmark results
bench_path = config.OUTPUT_DIR / 'benchmark_results.json'
if bench_path.exists():
    benchmark_results = json.loads(bench_path.read_text())
    print('✓ Benchmark results loaded')

print('\nEval results:', eval_results)
print('Benchmark results:', len(benchmark_results), 'variants')

✓ Eval results loaded from MLflow
✓ Benchmark results loaded

Eval results: {'finetuned_rouge1': 0.5683, 'finetuned_rougeL': 0.4028, 'finetuned_ragas_faithfulness': 0.6573}
Benchmark results: 3 variants


/home/ubuntu/Fine-tuning/.venv/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [5]:
# ── Step 3: Generate detailed model card ──────────────────────────────────────
# Pull live numbers from dataset + config so the card stays in sync as you retrain.
n_train  = sum(1 for _ in open(config.TRAIN_JSONL, encoding='utf-8'))
n_eval   = sum(1 for _ in open(config.EVAL_JSONL,  encoding='utf-8'))
total    = n_train + n_eval
epochs   = config.NUM_TRAIN_EPOCHS
eff_bs   = config.TRAIN_BATCH_SIZE * config.GRAD_ACCUM_STEPS

def _fmt(v):
    if isinstance(v, float):
        return f'{v:.4f}'
    return v if v not in (None, '') else '—'

rouge1       = _fmt(eval_results.get('finetuned_rouge1',          '—'))
rougeL       = _fmt(eval_results.get('finetuned_rougeL',          '—'))
faithfulness = _fmt(eval_results.get('finetuned_ragas_faithfulness', '—'))

bench_table = ''
if benchmark_results:
    bench_table = '\n| Variant | Memory (GB) | Latency (ms/tok) | ROUGE-L |\n|---------|-------------|------------------|---------|\n'
    for r in benchmark_results:
        if not r.get('error'):
            bench_table += f"| {r['name']} | {r['memory_gb']:.1f} | {r['latency_ms_per_token']:.1f} | {r['rouge_l']:.4f} |\n"

model_card = f'''---
base_model: mistralai/Mistral-7B-Instruct-v0.3
library_name: peft
pipeline_tag: text-generation
language:
- en
- hi
license: apache-2.0
tags:
- lora
- qlora
- sft
- legal
- indian-law
- mistral
---

# NyayaGPT — Mistral-7B Indian Legal Assistant

*Nyaya* (न्याय) — Sanskrit for **justice**

Fine-tuned Mistral-7B-Instruct-v0.3 on {total:,} Indian legal Q&A pairs via QLoRA.
Trained on IndianKanoon case law + GPT-4o synthetic augmentation.

## Quickstart

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base  = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    torch_dtype=torch.float16, device_map="auto"
)
tok   = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
model = PeftModel.from_pretrained(base, "{HF_USERNAME}/NyayaGPT-Mistral7B-adapter")
model.eval()

messages = [
    {{"role": "system", "content": "You are NyayaGPT, an expert Indian legal assistant."}},
    {{"role": "user",   "content": "What is Section 302 IPC?"}},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
ids    = tok(prompt, return_tensors="pt").to(model.device)
out    = model.generate(**ids, max_new_tokens=256, do_sample=False)
print(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))
```

## Training Details

| Attribute | Value |
|---|---|
| Base model | mistralai/Mistral-7B-Instruct-v0.3 |
| Method | QLoRA (LoRA r={config.LORA_R}, α={config.LORA_ALPHA}) |
| Training samples | {n_train:,} |
| Eval samples | {n_eval:,} |
| Epochs | {epochs} |
| Learning rate | {config.LEARNING_RATE:g} |
| Batch size | {config.TRAIN_BATCH_SIZE} × {config.GRAD_ACCUM_STEPS} grad accum = {eff_bs} effective |
| Hardware | NVIDIA RTX 5090 (32 GB) |
| Dataset | IndianKanoon case law + GPT-4o synthetic |

## Evaluation

| Metric | Score |
|--------|-------|
| ROUGE-1 | {rouge1} |
| ROUGE-L | {rougeL} |
| RAGAS Faithfulness | {faithfulness} |

## Quantization Benchmark
{bench_table if bench_table else '_Run notebook 04 to populate this section._'}

## Domain

Indian law: IPC, CrPC, Constitution, Contract Act, NI Act, IT Act, Civil Procedure Code,
property law, PIL, Supreme Court / High Court judgments.

## Disclaimer

This model is for educational and research purposes only. It does not constitute legal advice.
Always consult a qualified advocate for legal matters.
'''

# Write to adapters/README.md
card_path = config.ADAPTER_DIR / 'README.md'
card_path.write_text(model_card, encoding='utf-8')
print(f'✓ Model card written → {card_path}')
print(f'  samples: {n_train:,} train / {n_eval:,} eval | epochs: {epochs} | effective batch: {eff_bs}')
print()
print(model_card[:500])

✓ Model card written → /home/ubuntu/Fine-tuning/NyayaGPT/adapters/README.md
  samples: 1,521 train / 169 eval | epochs: 2 | effective batch: 8

---
base_model: mistralai/Mistral-7B-Instruct-v0.3
library_name: peft
pipeline_tag: text-generation
language:
- en
- hi
license: apache-2.0
tags:
- lora
- qlora
- sft
- legal
- indian-law
- mistral
---

# NyayaGPT — Mistral-7B Indian Legal Assistant

*Nyaya* (न्याय) — Sanskrit for **justice**

Fine-tuned Mistral-7B-Instruct-v0.3 on 1,690 Indian legal Q&A pairs via QLoRA.
Trained on IndianKanoon case law + GPT-4o synthetic augmentation.

## Quickstart

```python
from transformers import AutoToken


In [8]:
# ── Step 4: Push adapter to HF Hub ───────────────────────────────────────────
# Excludes the 493 MB of training checkpoints and any GGUF symlinks so the
# "adapter-only" repo stays ~80 MB.
from huggingface_hub import HfApi

if HF_USERNAME == 'YOUR_HF_USERNAME':
    print('⚠ Set HF_USERNAME in cell 2 before running this cell')
else:
    REPO_ID = f'{HF_USERNAME}/NyayaGPT-Mistral7B-adapter'
    api = HfApi(token=HF_TOKEN or None)
    api.create_repo(repo_id=REPO_ID, repo_type='model', exist_ok=True)
    api.upload_folder(
        folder_path=str(config.ADAPTER_DIR),
        repo_id=REPO_ID,
        repo_type='model',
        commit_message='Upload NyayaGPT LoRA adapter',
        ignore_patterns=['checkpoints/*', '*.gguf'],
    )
    print(f'✓ Adapter uploaded → https://huggingface.co/{REPO_ID}')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Adapter uploaded → https://huggingface.co/gauravgarwal/NyayaGPT-Mistral7B-adapter


In [9]:
# ── Step 5 (optional): Push merged FP16 model ─────────────────────────────────
# Merged FP16 weights already exist at /mnt/f/NyayaGPT-scratch/merged_model/
# (produced by scripts/build_gguf.py during Day 4 GGUF pipeline).
# This upload is ~14 GB — only do it if you want a self-contained standalone model.

MERGED_MODEL_DIR = Path('/mnt/f/NyayaGPT-scratch/merged_model')

if HF_USERNAME == 'YOUR_HF_USERNAME':
    print('⚠ Set HF_USERNAME in cell 2 before running this cell')
elif not MERGED_MODEL_DIR.exists():
    print(f'⚠ Merged FP16 model not found at {MERGED_MODEL_DIR}')
    print('  Run scripts/build_gguf.py first — it writes merged FP16 weights there.')
else:
    MERGED_REPO_ID = f'{HF_USERNAME}/NyayaGPT-Mistral7B'
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN or None)
    api.create_repo(repo_id=MERGED_REPO_ID, repo_type='model', exist_ok=True)
    api.upload_folder(
        folder_path=str(MERGED_MODEL_DIR),
        repo_id=MERGED_REPO_ID,
        repo_type='model',
        commit_message='Upload NyayaGPT merged FP16 model',
    )
    print(f'✓ Merged model uploaded → https://huggingface.co/{MERGED_REPO_ID}')

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

: 

## ✅ Day 6 Complete

**Model card:** `adapters/README.md`  
**HF Hub:** `https://huggingface.co/YOUR_USER/NyayaGPT-Mistral7B-adapter`

**Next:** Run `07_hf_spaces_demo.ipynb`